# Omnilingual ASR Pularr Fine-Tuning on Colab T4

This notebook runs the repeatable T4 workflow for Meta Omnilingual ASR CTC 300M on `google/WaxalNLP/ful_asr`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/zainmouhamed456-hub/Wispher_Pularr.git'
REPO_ROOT = '/content/whisper-pularr'
RUNS_ROOT = '/content/drive/MyDrive/omnilingual-pularr-runs'
DATASET_NAME = 'google/WaxalNLP'
DATASET_CONFIG = 'ful_asr'
OMNI_LANG = 'ful_Latn'
HF_HOME = '/content/hf-cache'
OMNI_BASELINE_SAMPLES = '64'
OMNI_SMOKE_STEPS = '100'
OMNI_MAIN_STEPS = '2000'
OMNI_MAX_DURATION_SECONDS = '40'

In [ ]:
import os
os.environ['HF_HOME'] = HF_HOME
os.environ['OMNI_LANG'] = OMNI_LANG
os.environ['OMNI_BASELINE_SAMPLES'] = OMNI_BASELINE_SAMPLES
os.environ['OMNI_SMOKE_STEPS'] = OMNI_SMOKE_STEPS
os.environ['OMNI_MAIN_STEPS'] = OMNI_MAIN_STEPS
os.environ['OMNI_MAX_DURATION_SECONDS'] = OMNI_MAX_DURATION_SECONDS
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
!mkdir -p "$HF_HOME" "$RUNS_ROOT"

In [ ]:
!nvidia-smi
!test -d "$REPO_ROOT" || git clone "$REPO_URL" "$REPO_ROOT"
%cd $REPO_ROOT
!git pull --ff-only || true

## Install and Verify Dependencies

Run this once per fresh Colab runtime.

In [ ]:
!bash colab/bootstrap_omnilingual_t4_free.sh "$REPO_ROOT" "$RUNS_ROOT"

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

repo_root = Path(REPO_ROOT)
runs_root = Path(RUNS_ROOT)
hf_home = Path(HF_HOME)
external_root = runs_root / 'external' / 'omnilingual-asr'

os.environ.setdefault('HF_HOME', str(hf_home))
os.environ.setdefault('PYTHONUNBUFFERED', '1')
os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
runs_root.mkdir(parents=True, exist_ok=True)
hf_home.mkdir(parents=True, exist_ok=True)

def add_omnilingual_paths():
    for path in (external_root / 'src', external_root):
        if path.exists() and str(path) not in sys.path:
            sys.path.insert(0, str(path))

def install_omnilingual_now():
    launcher = repo_root / 'colab' / 'run_omnilingual_t4_free.py'
    if launcher.exists():
        subprocess.check_call([
            sys.executable, str(launcher), 'bootstrap',
            '--runs-root', str(runs_root),
            '--hf-cache', str(hf_home),
        ], cwd=repo_root)
        return
    if not external_root.exists():
        subprocess.check_call([
            'git', 'clone', '--depth', '1',
            'https://github.com/facebookresearch/omnilingual-asr.git',
            str(external_root),
        ])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
        'torch==2.8.0', 'torchaudio==2.8.0',
    ])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        '--extra-index-url', 'https://fair.pkg.atmeta.com/fairseq2/whl/pt2.8.0/cu126',
        '-e', f'{external_root.as_posix()}[data]', 'jiwer==3.0.5',
    ])

add_omnilingual_paths()
try:
    import torch
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
except ModuleNotFoundError:
    install_omnilingual_now()
    add_omnilingual_paths()
    importlib.invalidate_caches()
    import torch
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

print('cuda_available=', torch.cuda.is_available())
print('gpu=', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('pipeline=', ASRInferencePipeline.__name__)

## Session 0: Baseline Selection

In [ ]:
!python colab/run_omnilingual_t4_free.py baseline --dataset-name "$DATASET_NAME" --dataset-config "$DATASET_CONFIG" --lang "$OMNI_LANG" --runs-root "$RUNS_ROOT" --hf-cache "$HF_HOME"

## Session 1: Prepare Waxal Parquet Dataset

In [ ]:
!python colab/run_omnilingual_t4_free.py prepare --dataset-name "$DATASET_NAME" --dataset-config "$DATASET_CONFIG" --lang "$OMNI_LANG" --runs-root "$RUNS_ROOT" --hf-cache "$HF_HOME"

## Session 2: Smoke Fine-Tune

In [ ]:
!python colab/run_omnilingual_t4_free.py train --runs-root "$RUNS_ROOT" --hf-cache "$HF_HOME" --smoke-steps 100

## Session 3: Main Fine-Tune

In [ ]:
!python colab/run_omnilingual_t4_free.py train --runs-root "$RUNS_ROOT" --hf-cache "$HF_HOME" --main-steps 2000 --learning-rate 1e-5 --grad-accumulation 16 --validate-every 250 --checkpoint-every 500

## Eval and Promotion

Set `EVAL_DIR` to the training output directory that contains `transcriptions/`.

In [ ]:
EVAL_DIR = ''
if EVAL_DIR:
    !python colab/run_omnilingual_t4_free.py eval --runs-root "$RUNS_ROOT" --hf-cache "$HF_HOME" --eval-dir "$EVAL_DIR"
    !python colab/run_omnilingual_t4_free.py promote --runs-root "$RUNS_ROOT" --hf-cache "$HF_HOME" --candidate-checkpoint "$EVAL_DIR"
else:
    print('Set EVAL_DIR to a run directory containing transcriptions before running promotion.')